## Overview
It involves 2 phases:

1. Shuffle Phase:
    - Both DataFrames are read and shuffled based on the join key
    - So that rows with the same key end up on the same partition

2. Hash Join Phase:
    - The smaller DataFrame (build side) is converted into a hash table in memory
    - The larger DataFrame (probe side) is streamed through, to perform Hash Join

## When It's Used
- Only for equi-joins '=' (joins based on equality conditions)
- When one DataFrame is small enough to fit in memory as a hash table but too large for broadcast
- When `spark.sql.join.preferSortMergeJoin` is set to `False`
- The join keys are not required to be sortable
- Can be forced using join hints

## When Not to Use
- When both DataFrames are large and cannot fit in memory
- When data is heavily skewed, leading to memory issues
- When the join condition is not an equality condition (<, >, <=, >=, etc.)

## Configuration
```python
# Disable sort merge join to prefer shuffle hash join
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
```

## Advantages
- Faster than Sort Merge Join when no sorting is needed
- More memory efficient than Broadcast Join for medium-sized datasets

## Disadvantages
- Requires shuffling data across the network (expensive)
- Build side must fit in memory
- Less efficient than Broadcast Join for small datasets

## Comparison with Other Joins
| Join Type | Shuffle Required | Memory Requirement | Best For |
|-----------|-----------------|-------------------|----------|
| Broadcast | No | Small (broadcast side) | Small datasets |
| Shuffle Hash | Yes | Medium (build side) | Medium datasets |
| Sort Merge | Yes | Low | Large datasets |

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [32]:
large_df = spark.range(1, 100000000).toDF("id")
large_df = large_df.withColumn("value", large_df.id * 2)

medium_df = spark.range(1, 100000).toDF("id")
medium_df = medium_df.withColumn("emp_id", (medium_df.id + 100).cast("string"))

In [35]:
print(spark.conf.get("spark.sql.join.preferSortMergeJoin"))

# Disable sort merge join to prefer shuffle hash join
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

print(spark.conf.get("spark.sql.join.preferSortMergeJoin"))


true
false


In [36]:
joined = large_df.join(medium_df.hint("shuffle_hash"), 'id')
joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#32L, value#34L, emp_id#37]
   +- ShuffledHashJoin [id#32L], [id#35L], Inner, BuildRight
      :- Exchange hashpartitioning(id#32L, 200), ENSURE_REQUIREMENTS, [plan_id=502]
      :  +- Project [id#32L, (id#32L * 2) AS value#34L]
      :     +- Range (1, 100000000, step=1, splits=2)
      +- Exchange hashpartitioning(id#35L, 200), ENSURE_REQUIREMENTS, [plan_id=503]
         +- Project [id#35L, cast((id#35L + 100) as string) AS emp_id#37]
            +- Range (1, 100000, step=1, splits=2)


